In [1]:
import numpy as np
list_files = np.load('octmnist.npz')
print(list_files)

val_images = list_files["val_images"]
val_labels = list_files["val_labels"]

NpzFile 'octmnist.npz' with keys: train_images, val_images, test_images, train_labels, val_labels...


In [2]:
val_images.shape
#10k instances
#28 features by 28 features grid

def reduce_ndims(array):
    '''Reduced dimensionality of a 3d square array by half'''

    x, y, z = array.shape

    reduced_array = np.zeros((x, int(y/2), int(z/2)))
    for i in range(x):
        for j in range(0, int(y/2) + 1, 2):
            for k in range(0, int(z/2) + 1, 2):
                p0 = array[i][j][k]
                p1 = array[i][j][k+1]
                p2 = array[i][j+1][k]
                p3 = array[i][j+1][k+1]

                new_point = (p0 + p1 + p2 + p3)/4

                reduced_array[i][int(j/2)][int(k/2)] = new_point
    
    return reduced_array


In [3]:
reduced_arr=reduce_ndims(val_images)


C:\Users\samue\AppData\Local\Temp\ipykernel_39692\3341804919.py:19: RuntimeWarning: overflow encountered in scalar add
  new_point = (p0 + p1 + p2 + p3)/4


In [4]:
val_images_reduced = reduce_ndims(reduced_arr)

val_images_reduced.shape

(10832, 7, 7)

In [5]:
val_images_2d = val_images_reduced.reshape(val_images_reduced.shape[0], -1)
val_images_2d.shape
#Reduced features to 49 features representing a 7x7 grid of grayscale images

def flatten_data(X):
    new_arr = X.reshape(X.shape[0], -1)
    
    return new_arr

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit

strata = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)

for train_index, test_index in strata.split(val_labels, val_labels):
    X_train = val_images[train_index]
    X_test = val_images[test_index]
    y_train = val_labels[train_index]
    y_test = val_labels[test_index]

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import FunctionTransformer, StandardScaler

flattener = FunctionTransformer(flatten_data)
scaler = StandardScaler()
svc = SVC()

pipe_baseline = Pipeline([
    ("flat", flattener),
    ("scale", scaler),
    ("model", svc)
])

In [7]:
reducer = FunctionTransformer(reduce_ndims)

from sklearn.decomposition import PCA

pca = PCA(n_components=50)

pipe_reduce = Pipeline([
    ("reduce1", reducer),
    ("reduce2", reducer),
    ("flat", flattener),
    ("scale", scaler),
    ("model", svc)
])

pipe_pca = Pipeline([
    ("flat", flattener),
    ("scale", scaler),
    ("pca", pca),
    ("model", svc)
])

pipe_both = Pipeline([
    ("reduce1", reducer),
    ("reduce2", reducer),
    ("flat", flattener),
    ("scale", scaler),
    ("pca", PCA(n_components=25)),
    ("model", svc)
])

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
import time

def train_eval_model(name, model):
    '''trains and evaluates the model'''
    t1 = time.time()
    model.fit(X_train, y_train)
    t2 = time.time()
    #get predictions
    pred = model.predict(X_test)
    
    t = t2 - t1
    acc = accuracy_score(pred, y_test)
    prec = precision_score(pred, y_test, average="weighted")
    recall = recall_score(pred, y_test, average="weighted")

    print(f"Model: {name}")
    print(f"Training Time: {t}")
    print(f'acc: {acc}')
    print(f'prec:{prec}')
    print(f'recall:{recall}\n')





In [11]:
train_eval_model("Baseline", pipe_baseline)
train_eval_model("Reduce", pipe_reduce)
train_eval_model("PCA", pipe_pca)
train_eval_model("Both", pipe_both)

c:\Users\samue\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\samue\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\samue\AppData\Local\Temp\ipykernel_39692\3341804919.py:19: RuntimeWarning: overflow encountered in scalar add
  new_point = (p0 + p1 + p2 + p3)/4


Model: Baseline
Training Time: 50.81444001197815
acc: 0.6972773419473927
prec:0.8329372340158594
recall:0.6972773419473927



c:\Users\samue\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\samue\AppData\Local\Temp\ipykernel_39692\3341804919.py:19: RuntimeWarning: overflow encountered in scalar add
  new_point = (p0 + p1 + p2 + p3)/4
c:\Users\samue\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model: Reduce
Training Time: 17.47329092025757
acc: 0.5288417166589755
prec:0.678801219843913
recall:0.5288417166589755



c:\Users\samue\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\samue\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\samue\AppData\Local\Temp\ipykernel_39692\3341804919.py:19: RuntimeWarning: overflow encountered in scalar add
  new_point = (p0 + p1 + p2 + p3)/4


Model: PCA
Training Time: 8.769004821777344
acc: 0.6751269035532995
prec:0.8161855904941941
recall:0.6751269035532995



c:\Users\samue\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\samue\AppData\Local\Temp\ipykernel_39692\3341804919.py:19: RuntimeWarning: overflow encountered in scalar add
  new_point = (p0 + p1 + p2 + p3)/4


Model: Both
Training Time: 16.372284412384033
acc: 0.5288417166589755
prec:0.678801219843913
recall:0.5288417166589755



c:\Users\samue\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
